### Pulling the data from the API

In [1]:
import sys
import os

# Add the current directory to the system path
# If your module is in a different directory, adjust the path accordingly
module_dir = os.path.abspath('.') 
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Verify the path was added (optional)
print(f"sys.path now includes: {module_dir}")

sys.path now includes: /Users/audracornick/Desktop/Herbarium Database/Final_dashboard


In [2]:
from specify_api.auth import make_session

# Make a new connection to the database
s = make_session()

# Get your own username
# safer request + debug output
# Debug the /context/user.json request and session cookies
r = s.get("/context/user.json", timeout=20, allow_redirects=True)
print("status:", r.status_code)
print("final url:", r.url)
print("redirect history:", [ (h.status_code, h.headers.get("location")) for h in r.history ])
print("content-type:", r.headers.get("content-type"))
print("session cookies:", [(c.name, c.domain, c.path) for c in s.cookies])
print("response preview:")
print(r.text[:2000])
try:
    me = r.json()
except ValueError:
    print("Response is not valid JSON; raw response:")
    print(r.text[:2000])
    raise
print(me.get("name"))

Getting credentials for https://database.beatymuseum.ubc.ca
status: 200
final url: https://database.beatymuseum.ubc.ca/context/user.json
redirect history: []
content-type: application/json
session cookies: [('collection', 'database.beatymuseum.ubc.ca', '/'), ('csrftoken', 'database.beatymuseum.ubc.ca', '/'), ('sessionid', 'database.beatymuseum.ubc.ca', '/')]
response preview:
{"id": 62, "accumminloggedin": null, "email": null, "isloggedin": false, "isloggedinreport": false, "logincollectionname": null, "logindisciplinename": null, "loginouttime": null, "name": "audra.cornick", "timestampcreated": "2026-02-27T15:02:48", "timestampmodified": "2026-02-27T15:02:48", "usertype": "LimitedAccess", "version": 1, "createdbyagent": "/api/specify/agent/1019158/", "modifiedbyagent": null, "agents": "/api/specify/agent/?specifyuser=62", "spappresources": "/api/specify/spappresource/?specifyuser=62", "spappresourcedirs": "/api/specify/spappresourcedir/?specifyuser=62", "spquerys": "/api/specify/spqu

In [3]:
# ─── Cell 1: Packages ─────────────────────────────────────────────────────────
from tqdm import tqdm
import pandas as pd
import time


In [4]:
# ─── Cell 2: Helper Functions ─────────────────────────────────────────────────
def extract_id(uri):
    """Pull the numeric ID out of a Specify resource URI string."""
    if not uri or not isinstance(uri, str):
        return None
    try:
        return int(uri.rstrip("/").split("/")[-1])
    except:
        return None


def extract_specimen_row(obj):
    """Flatten one collection object into a single row."""
    row = {}

    row["id"]            = obj.get("id")
    row["catalognumber"] = obj.get("catalognumber")

    # Collecting event
    ce = obj.get("collectingevent") or {}
    if isinstance(ce, str):
        ce = {}
    row["collection_date"] = ce.get("enddate")
    row["verbatim_date"]   = ce.get("enddateverbatim")

    # Locality - comes back as a URI string in bulk pulls
    loc = ce.get("locality")
    if isinstance(loc, str):
        row["locality_id"] = extract_id(loc)
    elif isinstance(loc, dict):
        row["locality_id"] = loc.get("id")
    else:
        row["locality_id"] = None

    # Collectors
    collectors       = ce.get("collectors") or []
    primary_agent_id = None
    agent_ids        = []

    for c in collectors:
        agent_uri = c.get("agent", "")
        agent_id  = extract_id(agent_uri) if isinstance(agent_uri, str) else None
        if agent_id:
            agent_ids.append(agent_id)
        if c.get("isprimary") and agent_id:
            primary_agent_id = agent_id

    if not primary_agent_id and collectors:
        first            = min(collectors, key=lambda c: c.get("ordernumber", 999))
        agent_uri        = first.get("agent", "")
        primary_agent_id = extract_id(agent_uri) if isinstance(agent_uri, str) else None

    row["all_agent_ids"]    = agent_ids
    row["primary_agent_id"] = primary_agent_id
    return row

In [5]:
# ─── Cell 3: Pull Collection Objects ──────────────────────────────────────────
# Update collectionmemberid when you switch to vascular plants
filter_str = (
    "&collectionmemberid=32768"
    "&collectingevent__isnull=False"
    "&determinations__isnull=False"
)

total = s.get(
    f"/api/specify/collectionobject/?limit=1&offset=0{filter_str}"
).json()["meta"]["total_count"]

print(f"Pulling {total:,} collection objects...")

raw_rows = []
limit    = 500

with tqdm(total=total, unit="rec") as pbar:
    offset = 0
    while offset < total:
        url     = f"/api/specify/collectionobject/?limit={limit}&offset={offset}{filter_str}"
        objects = s.get(url).json()["objects"]
        if not objects:
            break
        for obj in objects:
            raw_rows.append(extract_specimen_row(obj))
        pbar.update(len(objects))
        offset += limit
        time.sleep(0.05)

specimen_df = pd.DataFrame(raw_rows)

print(f"\nDone! {len(specimen_df):,} specimens")
print(f"Missing locality ID:       {specimen_df['locality_id'].isna().sum():,}")
print(f"Missing date:              {specimen_df['collection_date'].isna().sum():,}")
print(f"Missing primary collector: {specimen_df['primary_agent_id'].isna().sum():,}")

Pulling 105,027 collection objects...


100%|██████████████████████████████████| 105027/105027 [2:55:23<00:00,  9.98rec/s]



Done! 105,027 specimens
Missing locality ID:       24
Missing date:              4,692
Missing primary collector: 68


In [7]:
# ─── Cell 4: Pull All Localities and Filter ───────────────────────────────────
# id__in filter not supported - pull the full table and filter in pandas

total_locs = s.get(
    "/api/specify/locality/?limit=1&offset=0"
).json()["meta"]["total_count"]

print(f"Pulling {total_locs:,} localities...")

all_locs = []

with tqdm(total=total_locs, unit="rec") as pbar:
    offset = 0
    while offset < total_locs:
        url  = f"/api/specify/locality/?limit={limit}&offset={offset}"
        r    = s.get(url)
        locs = r.json().get("objects", [])
        if not locs:
            break
        for loc in locs:
            all_locs.append({
                "locality_id":  loc.get("id"),
                "localityname": loc.get("localityname"),
                "latitude":     loc.get("latitude1"),
                "longitude":    loc.get("longitude1")
            })
        pbar.update(len(locs))
        offset += limit
        time.sleep(0.05)

loc_df = pd.DataFrame(all_locs)

# Filter to only the localities linked to our specimens
locality_ids = specimen_df["locality_id"].dropna().unique()
loc_df       = loc_df[loc_df["locality_id"].isin(locality_ids)].copy()

print(f"\nTotal localities pulled: {total_locs:,}")
print(f"Localities linked to specimens: {len(loc_df):,}")
print(f"Missing coordinates: {loc_df['latitude'].isna().sum():,}")

Pulling 297,284 localities...


100%|██████████████████████████████████| 297284/297284 [1:13:45<00:00, 67.17rec/s]



Total localities pulled: 297,284
Localities linked to specimens: 22,747
Missing coordinates: 9,762


In [8]:
# ─── Cell 5: Pull Agents ──────────────────────────────────────────────────────
total_agents = s.get(
    "/api/specify/agent/?limit=1&offset=0&agenttype=1"
).json()["meta"]["total_count"]

print(f"Pulling {total_agents:,} agents...")

raw_agents = []

with tqdm(total=total_agents, unit="rec") as pbar:
    offset = 0
    while offset < total_agents:
        url     = f"/api/specify/agent/?limit={limit}&offset={offset}&agenttype=1"
        objects = s.get(url).json()["objects"]
        for obj in objects:
            raw_agents.append({
                "agent_id":      obj.get("id"),
                "firstname":     obj.get("firstname"),
                "lastname":      obj.get("lastname"),
                "middleinitial": obj.get("middleinitial")
            })
        pbar.update(len(objects))
        offset += limit
        time.sleep(0.05)

agent_df           = pd.DataFrame(raw_agents)
agent_df["full_name"] = (
    agent_df["firstname"].fillna("") + " " + agent_df["lastname"].fillna("")
).str.strip()

print(f"Done! {len(agent_df):,} agents")

Pulling 20,644 agents...


100%|██████████████████████████████████████| 20644/20644 [04:11<00:00, 82.12rec/s]

Done! 20,644 agents


In [9]:
# ─── Cell 6: Join Everything ──────────────────────────────────────────────────
df = specimen_df.merge(loc_df, on="locality_id", how="left")
df = df.merge(
    agent_df[["agent_id", "full_name", "lastname", "firstname"]],
    left_on="primary_agent_id",
    right_on="agent_id",
    how="left"
)

# Clean up
df["collection_date"] = pd.to_datetime(df["collection_date"], errors="coerce")
df["year"]            = df["collection_date"].dt.year
df["decade"]          = (df["year"] // 10) * 10
df["full_name"]       = df["full_name"].fillna("Unknown Collector")
df["localityname"]    = df["localityname"].fillna("Unknown Locality")
df["latitude"]        = pd.to_numeric(df["latitude"],  errors="coerce")
df["longitude"]       = pd.to_numeric(df["longitude"], errors="coerce")

print(f"Final dataset: {len(df):,} specimens")
print(f"Missing collector name: {df['full_name'].isna().sum():,}")
print(f"Missing date:           {df['collection_date'].isna().sum():,}")
print(f"Missing coordinates:    {df['latitude'].isna().sum():,}")

if df["year"].notna().any():
    print(f"Date range:             {int(df['year'].min())} – {int(df['year'].max())}")

print(f"Unique collectors:      {df['full_name'].nunique():,}")
print(f"\nTop 5 collectors:")
print(df["full_name"].value_counts().head())

Final dataset: 105,027 specimens
Missing collector name: 0
Missing date:           4,696
Missing coordinates:    33,045
Date range:             1689 – 2025
Unique collectors:      1,960

Top 5 collectors:
full_name
Sandra Lindstrom    11912
Jim Markham          9499
Tom Widdowson        9205
Daniel Guthrie       4319
Robert Scagel        4297
Name: count, dtype: int64


In [10]:
# ─── Cell 7: Save to CSV ──────────────────────────────────────────────────────
# Update the filename when you switch to vascular plants
output_file = "herbarium_algae.csv"

df.to_csv(output_file, index=False)

print(f"Saved {len(df):,} records to {output_file}")
print(f"Columns: {df.columns.tolist()}")

Saved 105,027 records to herbarium_algae.csv
Columns: ['id', 'catalognumber', 'collection_date', 'verbatim_date', 'locality_id', 'all_agent_ids', 'primary_agent_id', 'localityname', 'latitude', 'longitude', 'agent_id', 'full_name', 'lastname', 'firstname', 'year', 'decade']


In [11]:
# Find the collector with the suspiciously long span
suspicious = df.groupby("full_name").agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    specimen_count=("catalognumber", "count")
).reset_index()

suspicious["span"] = suspicious["last_year"] - suspicious["first_year"]
print(suspicious[suspicious["span"] > 100].sort_values("span", ascending=False))

            full_name  first_year  last_year  specimen_count   span
1787    Tom Widdowson      1690.0     2019.0            9205  329.0
368        Danny Pace      1689.0     1979.0            1889  290.0
1455  Paul Gabrielson      1787.0     2017.0             702  230.0
117         Anonymous      1835.0     2025.0             768  190.0
1591    Robert Scagel      1861.0     1986.0            4297  125.0
948       Jim Markham      1868.0     1979.0            9499  111.0
0                          1908.0     2017.0             457  109.0


In [13]:
# Then look at the specific specimens for that collector
name = "Tom Widdowson"
collector_specimens = df[df["full_name"] == name][
    ["catalognumber", "collection_date", "localityname", "latitude", "longitude"]
].sort_values("collection_date")
print(collector_specimens)

      catalognumber collection_date  \
40582       A012966      1690-06-16   
40583       A012966      1690-06-16   
40584       A012966      1690-06-16   
40651       A012965      1690-06-16   
40652       A012965      1690-06-16   
...             ...             ...   
96917       A011756             NaT   
96918       A011756             NaT   
98294       A099128             NaT   
98620       A004493             NaT   
99655       A099776             NaT   

                                            localityname   latitude  \
40582                    Gulf of Alaska, Adak, North Arm  51.783333   
40583                    Gulf of Alaska, Adak, North Arm  51.783333   
40584                    Gulf of Alaska, Adak, North Arm  51.783333   
40651                    Gulf of Alaska, Adak, North Arm  51.783333   
40652                    Gulf of Alaska, Adak, North Arm  51.783333   
...                                                  ...        ...   
96917  Loran station, spring islan

In [14]:
# Peek at collection objects without a collecting event
r = s.get("/api/specify/collectionobject/?limit=5&collectingevent__isnull=True&collectionmemberid=32768")
for obj in r.json()["objects"]:
    print(f"catalogeddate: {obj.get('catalogeddate')} | date1: {obj.get('date1')}")

catalogeddate: None | date1: None
catalogeddate: None | date1: None
catalogeddate: None | date1: None
